# Task 3 — Usage U2 full-RGB HOG linear SVM

Run All performs one fixed two-fold CPU screen. It uses a pure full-image RGB HOG descriptor, weighted scaling, a one-vs-rest linear SVM with `C=1`, and sigmoid calibration. There is no feature grid or model search.


## 1. Local runtime

Use the repository `.venv` kernel. This fixed-feature model runs on CPU and writes to the gitignored local Task 3 artifact folder.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def find_repo_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/fashion").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the cloned repository.")

REPO_DIR = find_repo_root()
expected_venv = (REPO_DIR / ".venv").resolve()
if Path(sys.prefix).resolve() != expected_venv:
    raise RuntimeError(f"Select the repository .venv kernel: {expected_venv}")
os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

LOCAL_TASK_DIR = REPO_DIR / "results/task3"
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"
LOCAL_EVIDENCE_DIR = REPO_DIR / "results/evidence/task3"
LOCAL_WORK_DIR = REPO_DIR / "tmp/task3-usage-hog"
print("Branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
teacher_dir = REPO_DIR / "data/raw/teacher"
required = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
    REPO_DIR / "data/processed/splits.csv",
    REPO_DIR / "data/processed/label_maps.json",
)
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Required teacher/processed files are missing: {missing}")
print("Official teacher data and canonical split are ready.")


## 2. Frozen hypothesis and gates

EDA found that pure full-RGB HOG was the strongest direct fixed-image probe for Usage (balanced diagnostic macro-F1 about 0.454). U2 asks whether replacing E2’s learned CNN representation with this lower-freedom shape/texture representation reduces memorisation while keeping rare-class signal. Labels, canonical folds, full canvas, effective-number weighting, and the E2 comparison rows stay fixed. No augmentation is used.

The screen advances by Route A only if macro-F1 is at least 0.417319 and the paired family-bootstrap lower 95% bound is above zero. Route B needs macro-F1 at least 0.402319 plus at least 25% clean-gap reduction or at least 10% NLL/Brier improvement. ECE must be at most 0.050, no supported class may lose more than 0.030 F1, and probability/split/resource checks must pass.


In [ ]:
USAGE_E2_PARENT_RUN_IDS = (
    "t3_usage_e2_class_balanced_ce_usage_smallcnn_f0_s2753_5461e048c3b3_20260830T115815Z356f6d",
    "t3_usage_e2_class_balanced_ce_usage_smallcnn_f1_s2753_5461e048c3b3_20260830T120645Z6d08cd",
    "t3_usage_e2_class_balanced_ce_usage_smallcnn_f2_s2753_5461e048c3b3_20260830T121514Zd58aa0",
    "t3_usage_e2_class_balanced_ce_usage_smallcnn_f3_s2753_5461e048c3b3_20260830T122347Z2cb06e",
    "t3_usage_e2_class_balanced_ce_usage_smallcnn_f4_s2753_5461e048c3b3_20260830T123218Z94db47",
)
USAGE_E2_ANCHOR = (
    LOCAL_EVIDENCE_DIR
    / "experiments/t3_usage_e2_class_balanced_ce/usage/aggregate/oof_predictions.csv"
)
if not USAGE_E2_ANCHOR.is_file():
    raise FileNotFoundError(
        f"Sync the E2 evidence first; the matched anchor is missing: {USAGE_E2_ANCHOR}"
    )

from fashion.train.task3_usage_hog_svm import check_usage_hog_svm_setup
preflight = check_usage_hog_svm_setup(root=REPO_DIR, folds=(0, 4))
print("Model:", preflight["model"])
print("Screen folds:", preflight["folds"])
print("Feature columns:", preflight["feature_columns"])
print("Estimated cache MiB:", round(preflight["estimated_cache_bytes"] / 1024**2, 1))
print("Model fits during check:", preflight["model_fits"])


## 3. Build or reuse the label-blind HOG cache

This is the long preparation step. It reads only official teacher images and does not fit a model.


In [ ]:
from fashion.train.task3_usage_hog_svm import prepare_usage_hog_features

prepared_features = prepare_usage_hog_features(
    root=REPO_DIR,
    output_root=LOCAL_TASK_DIR,
    workers=None,
    local_work_dir=LOCAL_WORK_DIR,
)
print(prepared_features["usage"])


## 4. Train folds 0 and 4

A matching completed fold is reused. Stop after this screen and analyse it before any five-fold run.


In [ ]:
from fashion.train.task3_usage_hog_svm import run_usage_hog_svm_screen

usage_u2 = run_usage_hog_svm_screen(
    prepared_features=prepared_features,
    parent_run_ids=USAGE_E2_PARENT_RUN_IDS,
    output_root=LOCAL_TASK_DIR,
    folds=(0, 4),
    registry_path=LOCAL_REGISTRY,
    root=REPO_DIR,
    anchor_prediction_path=USAGE_E2_ANCHOR,
    reuse_completed=True,
)
{
    "metrics_path": usage_u2["metrics_path"],
    "macro_f1": usage_u2["metrics"]["macro_f1"],
    "nll": usage_u2["metrics"]["nll"],
    "brier": usage_u2["metrics"]["brier"],
    "ece_15": usage_u2["metrics"]["ece_15"],
    "screen_gate": usage_u2["metrics"]["screen_gate"],
}


## 5. Stop

Do not train a second Usage model from this notebook.
